# 05 · Positions and loops (stages S3, S3b)

Can kept visual tokens keep their original positions, does the loop count control the layer calls, and is pass p of a
long run the same as the end of an R = p run?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import inspect
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
ROOT

In [ ]:
import torch

print("on inara:", "inara" in socket.gethostname(), "|", socket.gethostname())
[(torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1))
 for i in range(torch.cuda.device_count())]

In [ ]:
import pandas as pd

In [ ]:
MODEL_ID = cfg["model_id"]
RES_PX = cfg["res_px"]
PREFILL = cfg.get("answer_prefill")
MAX_NEW_TOKENS = cfg["max_new_tokens"]
R = 3
R_MAX = 5
N_ITEMS = 30
RUN_BATCH = True  # False: only one item in the full pass-state run
OUT_DIR = ROOT / cfg["out_dir"] / "nb05"
MODEL_ID, RES_PX, PREFILL, MAX_NEW_TOKENS, api.config_hash(cfg), api.git_hash()

### Step-1:- Load the model
Load once. Every later step reuses `vlm` and only changes the loop count at runtime.

In [ ]:
print("[STAGE] load model")
vlm = api.load_model(cfg)
language_model = vlm.model.model.language_model

In [ ]:
core = (language_model.config.loop_layer_index_start, language_model.config.loop_layer_index_end)
assert len(language_model.layers) == 24
assert core == (6, 18)
print(f"✓ model loaded, core layers {core[0]} to {core[1] - 1}")

### Step-2:- Read the modeling code
Where do positions come from? Look for `position_ids`, `relative_distance` (from `get_relative_distance`, which never
sees `position_ids`), `conv_mask`, and the loop over `loop_num_passes`. Findings: `docs/notes/modeling_code.md`.

#### Loop text model forward

In [ ]:
print(inspect.getsource(type(language_model).forward))

#### Relative distance

In [ ]:
print(inspect.getsource(inspect.getmodule(type(language_model)).get_relative_distance))

#### Attention forwards

In [ ]:
attention_classes = {type(layer.self_attn).__name__: type(layer.self_attn) for layer in language_model.layers}
list(attention_classes)

In [ ]:
for name, cls in attention_classes.items():
    print("#####", name)
    print(inspect.getsource(cls.forward))

### Step-3:- Prepare one item
One GQA dev item with all its visual tokens, used by the checks below.

In [ ]:
item = api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0]
item

In [ ]:
image = api.resize_long_side(item.image, RES_PX)
image

In [ ]:
inputs = api.prepare_inputs(vlm, api.item_prompt(item, cfg), image, prefill=PREFILL)
n_input = inputs["input_ids"].shape[-1]
n_input

In [ ]:
print("[STAGE] vision tower")
visual_tokens, vision_outputs = api.extract_visual_tokens(vlm, inputs)
n_vis = len(visual_tokens)
visual_tokens.shape

### Step-4:- Pruning keeping everything is bit-exact
Rebuilding the prompt with k = 100% must give exactly the same logits, otherwise pruning itself changes the model.

In [ ]:
api.set_loops(vlm, R)
full_logits = api.last_logits(vlm, inputs)
identity = torch.arange(n_vis)
full_pruned = api.prune_inputs(vlm, inputs, vision_outputs, identity)

In [ ]:
assert torch.equal(full_logits, api.last_logits(vlm, full_pruned))
print("✓ k=100% packed pruning is bit-exact")

### Step-5:- Original positions are not supported
`prune_inputs` packs the kept tokens. This step shows why there is no "positioned" variant: the model builds token
distances from the token index, so passing original `position_ids` should leave the logits unchanged.

#### Original positions of the kept tokens

In [ ]:
def original_positions(indices):
    """Unpruned prompt index of every token that survives pruning to `indices`."""
    ids = inputs["input_ids"][0]
    image_positions = (ids == vlm.model.config.image_token_id).nonzero(as_tuple=True)[0]
    keep = ids != vlm.model.config.image_token_id
    keep[image_positions[indices.to(ids.device)]] = True
    return keep.nonzero(as_tuple=True)[0][None]

In [ ]:
half, _ = api.text_mean_selector(visual_tokens, api.question_embeddings(vlm, item.question), 0.5)
packed = api.prune_inputs(vlm, inputs, vision_outputs, half)
original = original_positions(half)
default = torch.arange(n_input, device=original.device)[None]
original

In [ ]:
assert torch.equal(original_positions(identity), default)
assert original.shape[-1] == packed["input_ids"].shape[-1] < n_input
assert bool((original[0].diff() > 0).all()) and int(original.max()) < n_input
print(f"✓ k=100% positions are the default ones; k=50% keeps {original.shape[-1]} of {n_input}, in order")

#### Does the model use them?
The only reader of `position_ids` is a log-scaling in the full-attention layers, which is exactly 1 below position 8192.
Shifting past that shows the argument does arrive.

In [ ]:
packed_logits = api.last_logits(vlm, packed)
with_original = api.last_logits(vlm, {**packed, "position_ids": original})
shifted = api.last_logits(vlm, {**packed, "position_ids": original + 100_000})
(packed_logits - with_original).abs().max().item(), (packed_logits - shifted).abs().max().item()

In [ ]:
positions_unused = torch.equal(packed_logits, with_original)
assert torch.equal(full_logits, api.last_logits(vlm, {**full_pruned, "position_ids": default}))
assert positions_unused
assert not torch.equal(packed_logits, shifted)
print("✓ original position_ids change nothing at k=50%: pruning stays packed")

### Step-6:- Loop control
The prelude (6 layers) and coda (6) run once, the 12 core layers run R times, so a forward makes 6 + 12R + 6 layer calls.

#### One loop count

In [ ]:
api.set_loops(vlm, R)
calls = api.count_layer_calls(vlm, inputs)
calls

In [ ]:
assert calls == api.expected_layer_calls(vlm)
assert sum(calls) == 6 + 12 * R + 6
print(f"✓ R={R}: {sum(calls)} layer calls")

#### Every loop count

In [ ]:
print("[STAGE] loop control")
for r in range(1, R_MAX + 1):
    api.set_loops(vlm, r)
    calls = api.count_layer_calls(vlm, inputs)
    assert calls == api.expected_layer_calls(vlm)
    assert sum(calls) == 6 + 12 * r + 6
    print(f"✓ R={r}: {sum(calls)} layer calls")

In [ ]:
api.set_loops(vlm, 1)
logits_r1 = api.last_logits(vlm, inputs)
api.set_loops(vlm, 3)
assert not torch.equal(logits_r1, api.last_logits(vlm, inputs))
print("✓ logits differ between R=1 and R=3")

### Step-7:- Pass state on one item
The controller tree assumes that stopping at pass p equals a fresh run at R = p. `record_pass_states` keeps the last core
layer's output (layer 17) at the last prompt position, once per pass.

#### The long run

In [ ]:
api.set_loops(vlm, R_MAX)
with api.record_pass_states(vlm) as long_states:
    long_gen = api.generate(vlm, inputs, MAX_NEW_TOKENS)
[state.norm().item() for state in long_states]

#### A fresh run at R = 3

In [ ]:
api.set_loops(vlm, R)
with api.record_pass_states(vlm) as short_states:
    short_gen = api.generate(vlm, inputs, MAX_NEW_TOKENS)
short_gen.text, long_gen.text

In [ ]:
assert len(long_states) == R_MAX and len(short_states) == R
print(f"✓ pass {R}: max |difference| {(long_states[R - 1] - short_states[-1]).abs().max().item():.2e}")

### Step-8:- Pass state on a batch
The same comparison for every p and many dev items: first one item, then all.

#### What `answer_agree` means
The R = 5 run has no answer at pass p (the coda runs only after the last pass, and reading one out would need a custom
forward). So `answer_agree` is the agreement of the **R = p answer with the R = 5 answer**, counted where both parsed.
The p = 5 row repeats the same run, so it shows the rerun noise. If few answers parse, read `tokens_equal`.

In [ ]:
items = api.load_gqa(cfg, "dev", limit=N_ITEMS, root=ROOT)
len(items)

#### Sample: one item

In [ ]:
print("[STAGE] pass state sample")
sample = pd.DataFrame(api.check_pass_state(vlm, items[:1], cfg))
sample

In [ ]:
assert list(sample.p) == list(range(1, R_MAX + 1))
assert sample.cosine.notna().all()
print("✓ sample has one row per p")

#### Full run

In [ ]:
print("[STAGE] pass state")
batch = items if RUN_BATCH else items[:1]
rows = pd.DataFrame(api.check_pass_state(vlm, batch, cfg))
rows.head()

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = OUT_DIR / f"pass_state_n{len(batch)}_{api.config_hash(cfg)}_{api.git_hash()}.jsonl"
rows.to_json(out, orient="records", lines=True)
out

In [ ]:
assert len(rows) == R_MAX * len(batch)
print(f"✓ {len(rows)} rows saved")

#### Table

In [ ]:
table = rows.groupby("p").agg(
    n=("item_id", "size"), cosine_mean=("cosine", "mean"), cosine_min=("cosine", "min"),
    rel_l2_mean=("rel_l2", "mean"), rel_l2_max=("rel_l2", "max"), h_equal=("h_equal", "sum"),
    parsed_both=("parsed_both", "sum"), tokens_equal=("tokens_equal", "mean"),
    acc_R_p=("correct_p", "mean"), acc_R_5=("correct_full", "mean"))
table["answer_agree"] = rows[rows.parsed_both].groupby("p").answer_agree.mean()
table[["tokens_equal", "answer_agree", "acc_R_p", "acc_R_5"]] *= 100
table

### Step-9:- Decisions to log
Paste into `docs/decisions.md` and fill in the blanks.

In [ ]:
print(f"## <date> · S3 · token positions after pruning (git {api.git_hash()})")
print("- Decision: pruning stays packed. Inkling2Loop builds token distances, masks and convolutions from the token "
      "index, so original positions are not supported (docs/notes/modeling_code.md). "
      f"Original position_ids at k=50% leave the logits identical: {positions_unused}.")

In [ ]:
per_p = "; ".join(f"p={p}: min cos {r.cosine_min:.6f}, max rel L2 {r.rel_l2_max:.2e}, "
                  f"R=p vs R=5 answer agreement {r.answer_agree:.1f}% (n={r.parsed_both:.0f})"
                  for p, r in table.iterrows())
print(f"## <date> · S3b · loop control and pass state (git {api.git_hash()})")
print("- Layer calls = 6 + 12R + 6 for R=1..5; logits differ between R=1 and R=3.")
print(f"- Pass state, GQA dev n={len(batch)} (res_px={RES_PX}): {per_p}.")
print("- Decision: the pass-p state of a longer run <equals|differs from> the R=p run, so the tree assumption "
      "<holds|fails> because <reason>.")

### Step-10:- Figure F03

Pass p of the R = 5 run against a fresh R = p run, for every p: cosine similarity of the pass state and answer agreement. This is the check behind the tree assumption (a stop at pass p equals a run with R = p).

In [ ]:
# the table is indexed by p; the plot reads p as a column
pass_table = table.reset_index()
pass_table[["p", "n", "cosine_mean", "cosine_min", "answer_agree"]]

In [ ]:
api.set_paper_style()
fig = api.plot_pass_state(pass_table, R_MAX)

In [ ]:
pdf_path, png_path = api.save_figure(fig, "F03_pass_state", final=False)

In [ ]:
assert pdf_path.is_file() and png_path.is_file(), f"missing {pdf_path} or {png_path}"
print("✓ F03_pass_state draft saved as PDF and PNG (final=False, paper/ untouched)")